# Running Pre-Trained CGNet Arctic Model

Purpose:
--------
The purpose of this notebook is to run pre-trained CGnet Arctic models for machine learning detection of atmospheric rivers and tropical cyclones.\
See ClimateNet repo here: https://github.com/andregraubner/ClimateNet

Authors/Contributors:
---------------------
* Teagan King
* John Truesdale
* Katie Dagon

## Import libraries

In [1]:
import os
import sys
import numpy as np

sys.path.append("/glade/work/tking/cgnet/ClimateNet") # append path to ClimateNet repo
from climatenet.utils.data import ClimateDatasetLabeled, ClimateDataset
from climatenet.models import CGNet
from climatenet.utils.utils import Config
from climatenet.track_events import track_events
from climatenet.analyze_events import analyze_events
from climatenet.visualize_events import visualize_events

from os import path

## Confirm GPU resources
Can request through JupyterHub launch page.\
Current resources request (2/15/23): 1 node, 4 cpu, 64GB mem, 2 V100 GPU\
4/5/23: Noticed that not all CPU/mem was needed, new request: 1 node, 2 cpu, 64GB mem, 2 V100 GPU\
4/6/23: Trying out an increase in GPU to see if that improves speed: 1 node, 2 cpu, 64GB mem, 3 V100 GPU

In [2]:
# requires loading pytorch into environment
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
2


## Load pre-trained model
No need to specify config, just the folder with config/weights will work

In [3]:
cgnet = CGNet(model_path='/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.artic_052025_TMQ')

In [4]:
cgnet

In [5]:
cgnet.config

## Test inference on a subset of historical 2000 data

In [ ]:
inference = ClimateDataset('/glade/derecho/scratch/tking/cgnet/high_lat_QC/from_nersc/model_data_from_Katie/polar/nh/B20TRC5CN/', cgnet.config)

In [ ]:
inference.fields

In [ ]:
inference.length

In [ ]:
%%time
class_masks = cgnet.predict(inference) # masks with 1==TC, 2==AR

In [ ]:
class_masks

In [ ]:
# change the dataarray name
class_masks.name = 'masks'

In [ ]:
class_masks

In [ ]:
%%time
class_masks.to_netcdf("/glade/derecho/scratch/tking/cgnet/high_lat_QC/from_nersc/model_data_from_Katie/polar/nh/artic_masks.nc")

## Test inference on RCP2.6 2006 data

In [ ]:
inference_2006 = ClimateDataset('/glade/derecho/scratch/tking/cgnet/high_lat_QC/from_nersc/model_data_from_Katie/polar/nh/BRCP26C5CN/.....', cgnet.config) 

In [ ]:
inference_2006.length

In [ ]:
%%time
class_masks_2006 = cgnet.predict(inference_2006) # masks with 1==TC, 2==AR

In [ ]:
class_masks_2006.name = 'masks'
class_masks_2006.to_netcdf("/glade/derecho/scratch/tking/cgnet/high_lat_QC/from_nersc/model_data_from_Katie/polar/nh/BRCP26C5CN/2006/artic_masks_2006.nc")

## Test inference on subset of RCP8.5 2086 data

In [ ]:
inference_2100 = ClimateDataset('/glade/derecho/scratch/tking/cgnet/high_lat_QC/from_nersc/model_data_from_Katie/polar/nh/BRCP85C5CN/2100/final/', cgnet.config)

In [ ]:
inference_2100.length

In [ ]:
%%time
class_masks_2100 = cgnet.predict(inference_2100)

In [ ]:
class_masks_2100.name = 'masks'
class_masks_2100.to_netcdf("/glade/derecho/scratch/tking/cgnet/high_lat_QC/from_nersc/model_data_from_Katie/polar/nh/BRCP85C5CN/test_2100_artic_masks.nc")

## Function for creating TC/AR masks using pre-trained model

In [6]:
def cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False):
    """Use a pre-trained model to create masks of tropical cyclones (mask value = 1)
    and atmospheric rivers (mask value = 2)
    
    Function will create NetCDF mask files and save them in the save_directory.

    Parameters:
    -----------
    model_path: str
        filepath to pre-trained CGnet model
    inference_path: str
        filepath to inference data
    save_dir: str
        filepath to where the masks will be saved as .nc files
    analyze: bool (optional)
        default is False; if True, will save plots for analyzing events using climatenet.analyze_events().
        Note that this can significantly increase the time to run.
    visualize : bool (optional)
        default is False; if True, will save plots for visualizing events using climatenet.visualize_events().
        Note that this can significantly increase the time to run.
    
    """
    # instantiate CGNet with pre-trained model
    cgnet = CGNet(model_path=model_path) 
    
    # inference using the pre-trained config file
    inference = ClimateDataset(inference_path, cgnet.config)
    class_masks = cgnet.predict(inference) # masks with 1==TC, 2==AR

    # create save dir, if needed
    if not os.path.isdir(save_dir):
        os.makedirs(save_dir)
    else:
        print("Warning: might overwrite {}".format(save_dir))
    
    # save out class masks
    class_masks.name = 'masks'
    class_masks.to_netcdf(save_dir+"/class_masks.nc")
    print("Saved class masks to {}".format(save_dir))
    
    # note: this is resource intensive
    # event_masks = track_events(class_masks) # masks with event IDs
    # event_masks.to_netcdf(save_dir+"/event_masks.nc")
    # print("Saved event masks to {}".format(save_dir))
    
    if analyze:
        analyze_events(event_masks, class_masks, save_dir+"/")
        print("Analyze events done")
    if visualize:
        visualize_events(event_masks, inference, save_dir+"/")
        print("Visualize events done")

    return

## Pre-trained model which uses TMQ
each year takes ~9min to run

### CESM Historical output, 2000-2005

In [10]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.artic_052025_TMQ'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/B20TRC5CN/'

for year in range(2000, 2005):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/B20TRC5CN/2000/masks


100%|██████████| 730/730 [10:34<00:00,  1.15it/s]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/B20TRC5CN/2001/masks


100%|██████████| 730/730 [11:24<00:00,  1.07it/s]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/B20TRC5CN/2002/masks


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/B20TRC5CN/2003/masks


100%|██████████| 730/730 [09:09<00:00,  1.33it/s]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/B20TRC5CN/2004/masks
CPU times: user 13min 33s, sys: 5min 23s, total: 18min 57s
Wall time: 1h 20s


### CESM RCP2.6 output, 2006-2015

In [8]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.artic_052025_TMQ'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP26C5CN/'

for year in range(2014, 2016):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

100%|██████████| 730/730 [09:10<00:00,  1.33it/s]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP26C5CN/2014/masks


100%|██████████| 730/730 [08:59<00:00,  1.35it/s]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP26C5CN/2015/masks
CPU times: user 5min 18s, sys: 1min 35s, total: 6min 54s
Wall time: 19min 13s


In [11]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.artic_052025_TMQ'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP26C5CN/'

for year in range(2006, 2014):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP26C5CN/2006/masks


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP26C5CN/2007/masks


  0%|          | 0/730 [00:00<?, ?it/s]

...

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP26C5CN/2011/masks


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP26C5CN/2012/masks


100%|██████████| 730/730 [09:05<00:00,  1.34it/s]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP26C5CN/2013/masks
CPU times: user 21min 45s, sys: 6min 35s, total: 28min 21s
Wall time: 1h 16min 55s


### CESM RCP8.5 output, 2086-2100

In [9]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.artic_052025_TMQ'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh/BRCP85C5CN/'

for year in range(2086, 2101):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

100%|██████████| 729/729 [08:52<00:00,  1.37it/s]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh//BRCP85C5CN/2086/masks


100%|██████████| 730/730 [08:54<00:00,  1.37it/s]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh//BRCP85C5CN/2087/masks


  0%|          | 0/730 [00:00<?, ?it/s]

...

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh//BRCP85C5CN/2098/masks


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh//BRCP85C5CN/2099/masks


100%|██████████| 730/730 [08:49<00:00,  1.38it/s]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/nh//BRCP85C5CN/2100/masks
CPU times: user 39min 51s, sys: 10min 9s, total: 50min 1s
Wall time: 2h 20min 37s


# See combine_class_masks.ipynb in order to generate heat maps

# Pre-trained model which uses TMQ and V850

# Pre-trained model which uses TMQ, V850, U850, and PSL